# Tarea 1 - Métodos determinísticos para la física

### Emil Winkler Partida - A01352501
### 27/02/2026

## Método Newton - Rhampson



## Pregunta 2

 - ¿Que tipo de datos o clase tiene que ser cada uno de las entradas?
    - **function**: *Callable*. Acepta n argumentos numéricos escalares y devuelve un array
    - **jac_function**: *Callable*. Acepta n argumentos numéricos escalares y devuelve una matriz jacobiana n x n
    - **initial_value**: *Array*. Lista de números float con valores iniciales del problema.
    - **tolerance**: *Float*. Valor de referencia numérico para definir el error.
    - **max_iterations**: *Integer*. Número entero de la cantidad de iteraciones máximas.
 - En los casos en los que las entradas son funciones, ¿Que definición tienen estas funciones?
    - Primero se definen a nivel simbólico con *sympy* para que puedan operarse las derivadas exactas del jacobiano. No obstante, una vez hecho este proceso matemático se transforman a funciones numéricas que son computacionalmente más rápidas(*sp.lambdify*).
    - *function* es un vector columna con las ecuaciones a resolver, por otro lado, *jac_function* representa la matriz de derivadas de estas mismas funciones.
 - ¿Cual será tu condición de salida del ciclo iterativo?
    - Se utiliza el residuo, que es la norma del vector de la función (magnitud del vector de error) como marco de referencia. Si el ultimo residuo calculado es menor al valor definido en *tolerance*, el proceso iterativo se detiene con un break y significa que se ha encontrado la raiz. 


## Pregunta 3
### Función de Jacobiano y de Newton-Rhapson


In [1]:
# Importamos librerias importantes para el desarrollo del proyecto

import numpy as np
import sympy as sp
import math  

# Función para sacar Jacobiano para funciones de n dimensiones con n cantidad de variables

def jacobian(function, variables):
    jacobian_matriz = [] 
    for f in function:
        # Se saca la derivada de cada función con respecto  a cada variable con un for y ses agrega a la matriz del jacobiano
        jacobian_matriz.append([sp.diff(f,variables[i]) for i in range(len(variables))])  
    return jacobian_matriz

# Función del Método Newton-Raphson para resolver sistemas de ecuaciones no lineales

def newton_raphson(function, jac_function, initial_value, tolerance = 1e-10, max_iterations = 1000):
    residuals = [] # Inicializamos una lista para almacenar residuos

    # Creamos un arreglo que almacene los valores de las variables
    x_k = np.array(initial_value, dtype=float) 

    for _ in range(max_iterations):
        f_k = np.array(function(*x_k), dtype=float).flatten() # Utilizamos flattern para convertir la funcion en un arreglo unidimensional para operar
        J_k = np.array(jac_function(*x_k), dtype=float) # Calculamos el jacobiano de la función en el punto 

        # Resolvemos el sistema de ecuaciones lineales de J_k * delta = -f_k
        try:
            delta = np.linalg.solve(J_k, -f_k) 
        except np.linalg.LinAlgError: # Si el jacobiano es singular, no se puede resolver el sistema
            print("El Jacobiano es singular, no es posible continuar con el método.")
            return None, residuals
        
        # Actualizamos el valor de la variable en base al hiperplano tangente
        x_k = x_k + delta 

        # Calculamos el residuo como norma de la función en el punto actual y lo guardamos en la lista
        residuals.append(np.linalg.norm(f_k)) 

        # Si el ultimo residuo es menor a la tolerancia, se retorna la solución y los residuos
        if residuals[-1] < tolerance:
            break

    return x_k, residuals


## Pregunta 4
###  Encuentra el punto de intersección de las curvas $y = cos(x),   y = x^3 -1$

In [2]:
# Definimos variables arbitrarias para definir funciones a través de sympy
x, y = sp.symbols('x y')
variables = sp.Matrix([x, y])

# Definimos las funciones que queremos utilizar en el método de Newton-Raphson
f1 = sp.cos(x) - y
f2 = x**3 - 1 - y

# Creamos una matriz con las funciones definidas y calculamos el jacobiano
func_expresion = sp.Matrix([f1, f2])
jac_expresion = jacobian(func_expresion, variables)

# Convertimos expresiones simbólicas a funciones numéricas con lamdbify para poder evaluarlas
function = sp.lambdify((x, y), func_expresion, 'numpy') # Especificamos que queremos utilizar numpy para operaciones
jac_function = sp.lambdify((x, y), jac_expresion, 'numpy')

# Valores iniciales para el método
initial_value = [0.6, 0.4]

# Solucionamos las ecuaciones con newton-raphson
solution, residuals = newton_raphson(function, jac_function, initial_value)

print("Solución:", solution, "\nResiduos:", residuals[-1])

# Verificacion de la solución 
print("Comprobación final:")
valores_f = function(*solution).flatten() 
for i, val in enumerate(valores_f):
    print(f"f{i+1} = {val:.12f}")

# Como los valores de las funciones se aproximan a 0 significa que se encontro un punto donde se intersecectan las funciones

Solución: [1.12656191 0.42976673] 
Residuos: 4.47545209131181e-16
Comprobación final:
f1 = -0.000000000000
f2 = 0.000000000000


## Pregunta 5
### Encuentra el valor de ${x, y, z}$ que cumple con 
### $$z = 1 - 5e^-(\frac{x^2+y^2}{20})$$
### $$z = -x^2 - y^2$$
### $$ x + y = 1$$

In [ ]:
# Definimos variables arbitrarias para definir funciones a través de sympy
x, y, z = sp.symbols('x y z')
variables = sp.Matrix([x, y, z])

# Definimos las funciones que queremos utilizar en el método de Newton-Raphson
f1 = z - 1 + 5*sp.exp(-(x**2 +y**2)/20)
f2 = z + x**2 + y**2
f3 = x + y - 1

# Creamos una matriz con las funciones definidas y calculamos el jacobiano
func_expresion = sp.Matrix([f1, f2, f3])
jac_expresion = jacobian(func_expresion, variables)

# Convertimos expresiones simbólicas a funciones numéricas con lamdbify para poder evaluarlas
function = sp.lambdify((x, y, z), func_expresion, 'numpy') # Especificamos que queremos utilizar numpy para operaciones
jac_function = sp.lambdify((x, y, z), jac_expresion, 'numpy')

# Valores iniciales para el método
initial_value = [4, -3, -25]

# Solucionamos las ecuaciones con newton-raphson
solution, residuals = newton_raphson(function, jac_function, initial_value)

print("Solución:", solution, "\nResiduo final:", residuals[-1])

# Verificacion de la solución 
print("Comprobación final:")
valores_f = function(*solution).flatten()
for i, val in enumerate(valores_f):
    print(f"f{i+1} = {val:.12f}")

# Como los valores de las funciones se aproximan a 0 significa que se encontro un punto donde se intersecectan las funciones

Solución: [ 1.67261809 -0.67261809 -3.25006635] 
Residuo final: 2.808666774861361e-15
Comprobación final:
f1 = 0.000000000000
f2 = 0.000000000000
f3 = 0.000000000000


## Método Simpson


## Pregunta 6


- ¿Que tipo de datos o clase tiene que ser cada uno de las entradas?
    - **function**: *Callable*. Función matemática a integrar definidamente
    - **domain**: *List*. Contiene los elemenos *a* (float/int) =  límite inferior; *b* (float/int) = límite superior; *n* (int) = cantidad inicial de subintervalos.
    - **tolerance**: *Float*. Margen de error permitido 
- En los casos en los que las entradas son funciones, ¿Que definición tienen estas funciones?
    - En este caso, *function* representa una función a la cual se le aplicará la integral definida. Matemáticamente esta función debe ser capaz de recibir un argumento numérico y retornar un único valor numérico. Para esta aplicación se evaluara mucho esta función en base a iteraciones y el paso. 
- ¿Cuál será tu condición de salida del ciclo iterativo?
    - Este algoritmo calcula la integral con *n* intervalos y también la que tiene *2n* de subintervalos y las compara. Si el valor absoluto de la diferencia de estos resultados es menor a la *tolerance*, entonces se asume que el valor se estabilizo en una respuesta concreta y se cierra el ciclo while con un break.

## Pregunta 7
### Función para Método Simpson

In [ ]:

def integrate(function, domain, tolerance = 1e-6):
    a, b, n = domain # Separamos el dominio en sus componentes

    # Condición del método para la cantidad de subintervalos 
    if n % 2 != 0:
        print("n debe ser un número par para el método de Simpson")
        return
    
    while True:
        
        # Definimos "pasos" o subintervalos a través del dominio
        h = (b-a)/n

        # Definimos la cantidad de subintervalos para la comparativa de tolerancia a traves de 2n
        h2 = (b-a)/(2*n)

        # Asignamos valores de coeficientes
        coefficients = [1 if k == 0 or k == (n) else 2 if k % 2 == 0 else 4 for k in range(n+1)] # Usamos n+1 para incluir el punto final
        coefficients2 = [1 if k == 0 or k == (2*n) else 2 if k % 2 == 0 else 4 for k in range(2*n+1)] 
        
        # Defimos la suma/ponderación que compone al método
        total_sum = sum(coefficients[i]*function(a + i*h) for i in range(n+1))
        total_sum2 = sum(coefficients2[i]*function(a + i*h2) for i in  range(2*n+1))

        # Agregamos el factor de multiplicación a la suma de mi integral con n subintervalos y con 2n subintervalos
        integral1, integral2 = (h/3)*total_sum, (h2/3)*total_sum2

        # Hacemos comparativa |integral2 - integral1| < tolerancia para determinar la precisión
        if abs(integral2 - integral1) < tolerance:
            print(f"El valor de la integral: {integral2}, Subintervalos necesitados: {2*n}")
            break
        else:
            n *= 2

    return integral2    

El valor de la integral: 2.0000000645300013, Subintervalos necesitados: 64
Resultado: 2.0000000645300013


## Pregunta 8
### Área de un círculo unitario
### $$x^2 + y^2 = 1$$ 

In [78]:
# Función a integrar despejada para tener solo dependencia de x
function = lambda x: math.sqrt(1 - x**2) 

# Definimos el dominio del circulo unitario, con su n cantidad de subintervalos (Aunque estos puedan cambiar en base a la tolerancia)
domain = [-1, 1, 4]

# Calculamos la integral con el método de Simpson e imprimimos resultados (Notar que por la raiz solo tendremos la mitad positiva)
result = 2*integrate(function, domain, tolerance = 1e-6) # Multiplicamos por 2 para obtener el círculo completo
print(f"Resultado: {result}")

El valor de la integral: 1.570796017098502, Subintervalos necesitados: 16384
Resultado: 3.141592034197004


## Pregunta 9
### $$ \int_{-1}^{1} cosh(x)dx$$

In [81]:
# Función a integrar despejada para tener solo dependencia de x
function = lambda x: math.cosh(x) 

# Definimos el dominio del circulo unitario, con su n cantidad de subintervalos (Aunque estos puedan cambiar en base a la tolerancia)
domain = [-1, 1, 4]

# Calculamos la integral con el método de Simpson e imprimimos resultados (Notar que por la raiz solo tendremos la mitad positiva)
result = integrate(function, domain, tolerance = 1e-6) # Multiplicamos por 2 para obtener el círculo completo
print(f"Resultado: {result}")

El valor de la integral: 2.3504023997390346, Subintervalos necesitados: 64
Resultado: 2.3504023997390346


## Pregunta 10 
### $$\int_{-4\pi}^{4\pi} \frac{sin(x)}{x} dx$$

In [83]:
# Función a integrar despejada para tener solo dependencia de x
function = lambda x: math.sin(x)/x if x != 0 else 1 # Evitamos indeterminación para esta función

# Definimos el dominio del circulo unitario, con su n cantidad de subintervalos (Aunque estos puedan cambiar en base a la tolerancia)
domain = [-4*math.pi, 4*math.pi, 4]

# Calculamos la integral con el método de Simpson e imprimimos resultados (Notar que por la raiz solo tendremos la mitad positiva)
result = integrate(function, domain, tolerance = 1e-6) # Multiplicamos por 2 para obtener el círculo completo
print(f"Resultado: {result}")

El valor de la integral: 2.9843224462290197, Subintervalos necesitados: 512
Resultado: 2.9843224462290197


## Método Monte Carlo

## Pregunta 11

- ¿Que entradas debe tener la función? ¿Que tipo de datos o clase tiene que ser cada uno de las entradas? 
    - **function**: *Callable*. Función de N dimensiones/argumentos que devuelve un número al evaluarse
    - **limits**: *List*. Lista de tuplas con números reales (a,b) que define los límites para la región donde se tendran los puntos aleatorios.
    - **num_samples**: *Int*. Cantidad total de iteraciones. 
- ¿Cómo puedes generar una lista de números aleatorios? ¿Que tamaño tiene que tener esta lista?
    - Para hacer esto utilice un list comprehension para sacar puntos aleatorios con probabilidad uniforme entre el mínimo y el máximo de esa dimension. Cada vez que el ciclo en la lista de una vuelta se genera un solo punto. Básicamente la lista que genera un punto tiene una cantidad de n números para una n cantidad de dimensiones. No obstante en mi programa este proceso se repite por 1000000.
- ¿Cómo se define una hiperesfera? 
    - Como hablamos de un concepto generalizado y visualizado en 2D y 3D pero a N dimensiones, podemos definirlo como el conjunto de puntos en un espacio de n dimensiones que se encuentran a una distancia menor o igual a un radio *r* desde un punto central. 
    $$ x_1^2 + x_2^2 + . . . + x_n^2 \leq r^2$$
- ¿Que función tienes que integrar para encontrar su volumen?
    - Se integra una *función indicadora/característica* la cual sirve para definir si el dardo cayo dentro o fuera de la hiperesfera. Al promediar estos valores de 1 y 0 se multiplican por un volumen base y se calcula asi el volumen aproximado de la hiperesfera. 

    $$ f(x_1,...,x_n) = 1 \; si \;  x_1^2 + ... + x_n^2 \leq r^2$$
    $$ f(x_1,...,x_n) = 0 \;  si \;  x_1^2 + ... + x_n^2 > r^2$$


## Pregunta 12
### Función para Método Monte Carlo

In [ ]:
def sphere_volume(function, limits, num_samples = 1000000):
    # Limits es una lista de tuplas (min, max) para cada variable
    # Podemos saber las dimensiones del espacio con la longitud de la lista
    dimensions = len(limits)
    print(f"Dimensiones del espacio: {dimensions}")

    volumen = 1 # Inicializamos el volumen del espacio del hiper-volumen
    for a, b in limits:
        if a >= b:
            print("Los limites deben ser válidos (min < max)") # a debe ser menor que b para que el espacio tenga volumen 
            return None
        volumen *= (b-a) # Calculamos el volumen del espacio en base a los limites de integración 
    
    sum_dim = 0 # Inicializamos la suma donde se evaluará la función en los puntos aleatorios generados
    for _ in range(num_samples):
        # Generamos un punto aleatorio dentro de los limites de integración para cada dimensión
        rand_point = [np.random.uniform(a, b) for a, b in limits]
        # Evaluamos la función en ese punto para sumarla a un total
        sum_dim += function(*rand_point)

    # Retornamos el resultado de la integral, que se define como el promedio de las evaluaciones por el volumen del espacio
    return (sum_dim/num_samples) * volumen


## Pregunta 13
### Área de un círculo 

In [95]:
# Definicmos funcion indicadora para el interior de la esfera unitaria
def radius(x, y):
    return 1 if x**2 + y**2 <=1 else 0

# Damos los limites de integración para cada una de las variables. 
limits = [(-1,1),(-1,1)]

# Calculamos el área de nuestro círculo unitario para poder comparar con el resultado anterior por el método de Simpson
area_circle = sphere_volume(radius, limits)

print(f"Área aproximada del círculo unitario: {area_circle}")

Dimensiones del espacio: 2
Área aproximada del círculo unitario: 3.14628


## Pregunta 14
### Volumen de una hiperesfera

In [ ]:
# Definicmos funcion indicadora para el interior de la hiperesfera con limites de radio 1
def hyper_radius(*coordinates):
    return 1 if sum(x**2 for x in coordinates) <= 1 else 0

# Damos los limites de integración para cada una de las variables. En este caso lo haremos de 4D 
limits = [(-1,1),(-1,1),(-1,1),(-1,1)]

# Calculamos el volumen aproximado de una hiperesfera de radio 1 en 4 dimensiones
volume_hypersphere = sphere_volume(hyper_radius, limits)

print(f"Volumen de hyperesfera de radio 1 en 4D: {volume_hypersphere}")


Dimensiones del espacio: 4
Volumen de hyperesfera de radio 1 en 4D: 4.92912
